# 02. Baseline Linear Modeling & Hyperparameter Search
> **Benchmarking regularized Logistic Regression vs. Naive Baseline**

Part of the **GreenlightML** open-source film risk modeling and portfolio optimization engine.


## 1. Environment & Feature Preparation
Load clean data and extract pre-release feature matrices.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier

sys.path.append(os.path.abspath('..'))
from src.data import load_and_merge_data, filter_clean_theatrical
from src.features import build_features
from src.models import get_logistic_regression
from src.metrics import evaluate_model

df_clean = filter_clean_theatrical(load_and_merge_data())
X_pre, X_leak, y = build_features(df_clean)

# Stratified 80/20 train/test holdout split
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_pre, y, df_clean.index, test_size=0.2, random_state=42, stratify=y
)
test_slice = df_clean.loc[idx_test]

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)


## 2. Benchmark: Naive Dummy Classifier
Demonstrate why single-point accuracy is misleading on skewed classes.


In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train_sc, y_train)
dummy_pred = dummy.predict(X_test_sc)

res_dummy = evaluate_model("Naive Dummy Baseline", "Passive", y_test, dummy_pred, None, test_slice)
pd.DataFrame([res_dummy])


## 3. Model 1: Logistic Regression & Systematic Hyperparameter Tuning
Vary regularization strength C and class weighting using Stratified 5-Fold Cross-Validation.


In [ ]:
param_grid = {
    'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
    'class_weight': [None, 'balanced']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(get_logistic_regression(), param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train_sc, y_train)

print(f"Best CV ROC-AUC: {grid.best_score_:.4f}")
print(f"Best Hyperparameters: {grid.best_params_}")


## 4. Out-of-Sample Holdout Evaluation
Evaluate tuned Logistic Regression against unseen test titles.


In [ ]:
best_lr = grid.best_estimator_
y_pred_lr = best_lr.predict(X_test_sc)
y_prob_lr = best_lr.predict_proba(X_test_sc)[:, 1]

res_lr = evaluate_model("Logistic Regression (Tuned)", "Pre-release", y_test, y_pred_lr, y_prob_lr, test_slice)
pd.DataFrame([res_dummy, res_lr])
